# What Should an Agent Remember? — Experiment Runner
**MA981 dissertation submission notebook (executable, FASER requirement).**

Runs the complete comparative experiment: one fixed LLM x five memory
architectures (B, M1–M4) x two evaluation modes, then the statistical
analysis and figures. Every step is resumable and seeded; the code lives in
`src/agentmem/` (submitted alongside as machine-readable PDF).

Run order matters — execute top to bottom. With `provider: mock` in
`configs/base.yaml` this completes on any CPU laptop with zero cost; with
`provider: university_hpc` it performs the real run (see docs/HPC_RUN_GUIDE.md).

In [ ]:
# Environment check + imports
import sys, platform, pathlib
ROOT = pathlib.Path.cwd()
assert (ROOT / "src" / "agentmem").exists(), "Run this notebook from agent-memory/"
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
print(platform.platform(), "| python", sys.version.split()[0])
import agentmem, yaml
CONFIG = yaml.safe_load(open("configs/base.yaml"))
print("provider:", CONFIG["provider"], "| model:", CONFIG["model"] or "(mock)",
      "| DP1 budget:", CONFIG["budgets"]["memory_read_tokens"], "tokens",
      "| seeds:", CONFIG["seeds"])

## 1. Verification suite
All architecture/budget/metric invariants (incl. DP1: no memory may exceed the shared read budget).

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, "tests failed"

## 2. Full experiment grid
Every seed x task x condition. Resumable: completed (condition, item, seed) cells are skipped, so interrupting and re-running is safe.

In [ ]:
r = subprocess.run([sys.executable, "scripts/run_full_grid.py"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-500:] if r.returncode else "")
assert r.returncode == 0

## 3. Headline results

In [ ]:
from agentmem.logging.trajectory import load_records
from analysis.stats import records_to_df, summary_table, failure_crosstab
import pathlib
for f in sorted(pathlib.Path("results/runs").glob("*.jsonl")):
    df = records_to_df(load_records(f))
    if df.empty: continue
    print(f"\n=== {f.stem} ===")
    display(summary_table(df))
    display(failure_crosstab(df))

## 4. Inferential statistics
Wilcoxon signed-rank vs baseline (Holm-corrected) + bootstrap CIs. Full battery incl. logistic regression is written to `results/report.md`.

In [ ]:
from analysis.stats import wilcoxon_contrast, holm_correct, bootstrap_diff_ci
for f in sorted(pathlib.Path("results/runs").glob("*.jsonl")):
    df = records_to_df(load_records(f))
    if df.empty: continue
    present = [c for c in ["m1","m2","m3","m4"] if c in set(df.condition)]
    fam = holm_correct([wilcoxon_contrast(df, c, "b") for c in present])
    print(f"\n=== {f.stem}: vs baseline ===")
    for r in fam:
        ci = bootstrap_diff_ci(df, r["contrast"].split()[0], "b")
        print(f"{r['contrast']:<11} diff={r['mean_diff']:+.3f} "
              f"CI[{ci['ci_low']:+.3f},{ci['ci_high']:+.3f}] "
              f"p_holm={r['p_holm']:.4f} {'SIG' if r['significant_05'] else 'ns'}")

## 5. Figures

In [ ]:
from IPython.display import Image, display
for p in sorted(pathlib.Path("results/figures").glob("*.png")):
    print(p.name); display(Image(str(p), width=640))

## 6. Reproducibility manifest
The exact config, git commit and platform of this run (also at `results/runs/manifest.json`).

In [ ]:
import json
print(json.dumps(json.load(open("results/runs/manifest.json")), indent=2)[:2000])